# Canvas LMS Automation Test Notebook

This notebook demonstrates how to automate the following after successful payment:

1. Create student in Canvas LMS
2. Enroll student in selected course
3. Generate credentials
4. Store LMS mapping locally

**Before running:**
- Set your Canvas domain (e.g., https://ayatech.org)
- Provide a valid Canvas API token
- Provide a valid Canvas course ID


In [ ]:
# Install requirements if needed
# !pip install requests

In [ ]:
import requests
import uuid
import json
from datetime import datetime

# =============================
# CONFIGURATION (EDIT THESE)
# =============================

CANVAS_DOMAIN = "https://ayatech.org"  # e.g., https://ayatech.org
CANVAS_API_TOKEN = "YOUR_CANVAS_API_TOKEN"
COURSE_ID = "YOUR_COURSE_ID"

HEADERS = {
    "Authorization": f"Bearer {CANVAS_API_TOKEN}",
    "Content-Type": "application/json"
}

print("Config loaded")

## Helper: Generate Canvas Credentials

In [ ]:
def generate_credentials(student_email):
    username = student_email
    password = uuid.uuid4().hex[:10]
    return username, password

print("Credential generator ready")

## Step 1: Create Student in Canvas

In [ ]:
def create_canvas_user(name, email, password):
    url = f"{CANVAS_DOMAIN}/api/v1/accounts/self/users"

    payload = {
        "user": {
            "name": name,
            "short_name": name,
            "sortable_name": name
        },
        "pseudonym": {
            "unique_id": email,
            "password": password
        },
        "communication_channel": {
            "type": "email",
            "address": email
        }
    }

    response = requests.post(url, headers=HEADERS, json=payload)
    response.raise_for_status()
    return response.json()

print("Create user function ready")

## Step 2: Enroll Student in Course

In [ ]:
def enroll_student(course_id, user_id):
    url = f"{CANVAS_DOMAIN}/api/v1/courses/{course_id}/enrollments"

    payload = {
        "enrollment": {
            "user_id": user_id,
            "type": "StudentEnrollment",
            "enrollment_state": "active"
        }
    }

    response = requests.post(url, headers=HEADERS, json=payload)
    response.raise_for_status()
    return response.json()

print("Enrollment function ready")

## Step 3: Store LMS Mapping (Local JSON for testing)

In [ ]:
def store_lms_mapping(student_email, canvas_user_id, course_id):
    record = {
        "student_email": student_email,
        "canvas_user_id": canvas_user_id,
        "course_id": course_id,
        "timestamp": datetime.utcnow().isoformat()
    }

    try:
        with open("lms_mappings.json", "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        data = []

    data.append(record)

    with open("lms_mappings.json", "w") as f:
        json.dump(data, f, indent=2)

    return record

print("Mapping storage ready")

## 🚀 Full Automation Test (Run This)

In [ ]:
# ===== TEST INPUT =====
student_name = "Test Student"
student_email = f"test_{uuid.uuid4().hex[:6]}@example.com"

# Step 1: Generate credentials
username, password = generate_credentials(student_email)
print("Generated credentials:", username, password)

# Step 2: Create Canvas user
user_response = create_canvas_user(student_name, student_email, password)
canvas_user_id = user_response["id"]
print("Canvas user created:", canvas_user_id)

# Step 3: Enroll in course
enroll_response = enroll_student(COURSE_ID, canvas_user_id)
print("Enrollment successful")

# Step 4: Store mapping
mapping = store_lms_mapping(student_email, canvas_user_id, COURSE_ID)
print("Mapping stored:", mapping)

print("\n✅ Automation completed successfully")